# H10 — 并发是否真的解决你的问题

**学习入口：把下面 prompt 发给 VS Code Codex 侧边栏。** Notebook 也可自行逐格阅读。
C++ 在持续 ESP-IDF 工程编辑与构建；Notebook 运行仅显示示例/手动证据卡。
**状态：Alpha；固件构建与实机验收分开记录，见 ../VALIDATION.md。**

```text
开始 H10 — 并发是否真的解决你的问题。
请先读取本工程 AGENTS.md、.academy/course.json 和 .academy/progress.md。
通过 course_root 找到 step2/course.json 和 step2/notebooks/H10-rtos.ipynb。
读取 Notebook 的 cells/source 与 academy.step_id/mode；先读当前步骤相关格。
检查工程、环境和前置条件，从第一个未完成步骤开始；每轮只推进一个小步骤。
Guided 可合并指定示例并说明 diff；Modify/Debug/Build/Open 先让我尝试，按需逐级提示。
不要主动读取 solution.md 或其他答案。环境故障可以排查修复，解释后回到原步骤。
不要把挑战 Bug 自动修掉。以当前代码与真实证据验收，记录提示级别和下一步。
我已有一定 C++ 基础，Python 几乎零基础。现在开始。
```

不用 agent 时按步骤标题顺序操作；卡住后再逐级查看本课 hints.md。启动 prompt 的路径按工作区课程链接解析，不靠其他窗口或聊天记忆。

## 我们要解决什么问题？

传感器、网络和按钮一起工作时开始卡顿。先测瓶颈，再用 Queue 分离工作，避免用更多 Task 制造更多竞态。

前置：H09 主线功能及性能记录。

## 本次够用的知识

ESP-IDF 已在 FreeRTOS 上运行，但业务不必多 Task。Queue 拷贝 item 字节，不会深拷贝 C++ 动态对象；Mutex 保护共享资源，Semaphore 表达事件/数量；EventGroup 表示状态位，软件 Timer callback 应短。ESP-IDF 的 Task 栈参数以 byte 为单位，与 vanilla FreeRTOS 文档区分。

先预测、运行和记录，再补概念。每节开始保存现有工程差异，后续沿用同一工程。

## 可选：Notebook 示例与证据卡

使用课程 .venv kernel 运行下格；这只是显示工具，不需要学习 Python。侧边栏学习不依赖这些格，build/flash/monitor 都在 ESP-IDF Terminal 执行。

In [ ]:
from pathlib import Path
import sys
import json

course_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "step2/course.json").is_file():
        course_root = candidate
        break
    link = candidate / ".academy/course.json"
    if link.is_file():
        course_root = Path(json.loads(link.read_text(encoding="utf-8"))["course_root"])
        break
if course_root is None or not (course_root / "step2/course.json").is_file():
    raise RuntimeError("请从课程工作区运行，或先在学习工程 .academy/course.json 链接课程根目录")
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))
from tools.esp32_notebook import install, hardware_checklist
install(get_ipython())


## 先测单循环 · Guided

在 H09 工程同时采样和发送状态，记录按钮响应时间、网络延迟、heap 与 Task stack watermark。找出可被移出的长操作。
如果短事件循环已满足预算，可以保留它；本课 Queue 实验帮助理解边界，不要求工具架构必须多任务。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 一个 Producer，一个 Owner · Guided

REQUIRES freertos log。示例 Sample 只包含整数，Queue 容量 4；producer 每 100 ms 发送，owner 消费。检查创建失败与发送 timeout，不忽略返回值。
在持续工程中新增 sampler task，并把业务状态保留在唯一 owner；不是粘贴第二套入口。IDF stack watermark 单位按实际 IDF 文档核对。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

示例来源：`step2/lessons/H10/examples/queue.cpp`。目标：`main/main.cpp`。这是教学示例；合并后以工程源码为准，后续不要来回修改两份实现。依赖参考：`REQUIRES freertos log`；SRCS 按本步骤目标文件登记。

In [ ]:
%%esp32_example --target main/main.cpp
#include <cstdint>
#include "esp_log.h"
#include "freertos/FreeRTOS.h"
#include "freertos/task.h"
#include "freertos/queue.h"

struct Sample {
    uint32_t sequence;
    int value;
};
static QueueHandle_t samples;
static void producer(void*) {
    uint32_t sequence = 0;
    while (true) {
        Sample sample{sequence++, 42};
        if (xQueueSend(samples, &sample, pdMS_TO_TICKS(20)) != pdTRUE) {
            ESP_LOGW("queue", "full: dropped sequence=%lu",
                     (unsigned long)sample.sequence);
        }
        vTaskDelay(pdMS_TO_TICKS(100));
    }
}
extern "C" void app_main() {
    samples = xQueueCreate(4, sizeof(Sample));
    if (!samples) {
        ESP_LOGE("queue", "allocation failed");
        return;
    }
    if (xTaskCreate(producer, "sampler", 3072, nullptr, 5, nullptr) != pdPASS) {
        vQueueDelete(samples);
        ESP_LOGE("queue", "task creation failed");
        return;
    }
    Sample received{};
    while (true) {
        if (xQueueReceive(samples, &received, pdMS_TO_TICKS(500)) == pdTRUE) {
            ESP_LOGI("queue", "sequence=%lu value=%d stack_min_bytes=%u",
                     (unsigned long)received.sequence, received.value,
                     (unsigned)uxTaskGetStackHighWaterMark(nullptr));
        }
    }
}


## 队列满的行为必须可见 · Modify

让 consumer 每次处理后等待 500 ms，观察 sequence 缺口与 full 日志。恢复，再比较丢最新、保留最新或背压三种策略。
业务“按钮取消”不能与可丢采样同等处理：定义优先规则与容量，不靠加大队列掩盖无限输入。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 选择同步工具 · Build

分别实现三个小需求：等待一次采样完成（Semaphore）、保护共享总线（Mutex）、等待 network-ready 状态（EventGroup）。每次解释为何不直接用同一种工具。
软件 Timer 仅向 owner 送事件。Mutex 不是 ISR 可用机制；跨线程传 std::string 指针要有明确生命周期，不用 Queue 直接拷贝 std::string 对象。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 资源顺序造成卡死 · Debug

先在纸上分析 A 先锁 bus 再锁 state、B 先锁 state 再锁 bus，指出循环等待。设计统一锁顺序，或用 owner task 消除共享。
将 producer 的 delay 暂时去掉，观察负载与丢样，再恢复；不要先提升优先级或关闭 watchdog。记录 stack、阻塞和调度证据。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 证明忙的时候仍能取消 · Build

需求：网络发送变慢/传感器超时，按钮取消仍达到你的响应预算；队列满有可见统计；资源只由 owner 管理；临时字符串无悬空引用。
在 host 做消息/溢出测试，在真实 FreeRTOS 负载下测响应。Mutex、Semaphore、Event、Timer 只加入需要的部分。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 删掉一个不必要的 Task · Open

回顾每个 Task 的职责、周期、stack、priority 和资源，把没有独立等待需求的 Task 合并回事件循环，比较前后行为。
H11 的存储写入也有成本，不要在每次 tick 或高频回调写 Flash。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 验收与复盘

- [ ] 队列满/创建失败有处理，消息无生命周期漏洞
- [ ] 真实负载下按钮响应预算有测量
- [ ] 能解释为何使用或不使用多 Task

分别记录 host、firmware build、flash、日志、实物与网络往返；未知保持未确认。手动卡片只是学习者记录，不自动测量，也不自动写入进度文件。

复盘：我选择了什么机制？哪个证据排除了错误假设？哪些内容还需要测量？把当前 Step、Hint level、代码版本、证据和下次起点写入 `.academy/progress.md`。

In [ ]:
hardware_checklist(
    'H10',
    [
        '队列满/创建失败有处理，消息无生命周期漏洞',
        '真实负载下按钮响应预算有测量',
        '能解释为何使用或不使用多 Task',
    ],
)


## 遇到困难时

先完成一次尝试，再打开 `step2/lessons/H10/hints.md`，每次只读当前步骤的一层。完整参考与解题分析单独保存，默认路径不主动读取。

## 官方版本参考

- [api-reference/system/freertos_idf.html](https://docs.espressif.com/projects/esp-idf/en/v5.5.1/esp32/api-reference/system/freertos_idf.html)